# Maize Tassel Detection - YOLO26s Training (Colab)
**FYP-26-S2-7 | MTDC-UAV Dataset | Adapted Notebook**

## What This Notebook Does
1. Extracts MTDC-UAV drone images with VOC XML annotations
2. Converts annotations to YOLO format
3. Splits large UAV images into 640x640 tiles (SAHI)
4. Trains YOLO26s in 3 stages (warmup -> full -> high-res)
5. Saves best model weights (best.pt) to Google Drive

## Before You Start
- Upload `MTDC-UAV.zip` to your Google Drive root folder
- Training takes ~4 hours on Tesla T4 GPU

## Open In Colab
`Runtime -> Change runtime type -> T4 GPU`


In [ ]:
import os, sys, subprocess

# 1. Check GPU
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu_info.stdout[:600])
if gpu_info.returncode != 0:
    raise SystemExit("No GPU detected. Enable GPU: Runtime -> Change runtime type -> T4 GPU")

# 2. Install packages
!pip install -q ultralytics scipy pillow opencv-python matplotlib pyyaml 2>&1 | tail -3
print("All packages installed - ready to go")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# === CONFIGURATION (edit these if your files differ) ===
UAV_ZIP_PATH   = '/content/drive/MyDrive/MTDC-UAV.zip'
DRIVE_RESULTS  = '/content/drive/MyDrive/maize_yolo26_results'

# Model config
MODEL_SIZE     = 'yolo26s'
IMG_SIZE       = 640
BATCH          = 16
NMS_IOU        = 0.4
TRAIN_IOU      = 0.7
EVAL_CONF_THR  = 0.30
RUN_NAME       = 'maize_yolo26'

# Work directories
EXTRACT_PATH   = '/content/dataset'
WORK_DIR       = '/content/maize_work'
DATASET_DIR    = os.path.join(WORK_DIR, 'dataset')
TILED_DIR      = os.path.join(WORK_DIR, 'uav_tiled')

# Stage names
S1_NAME = f'{RUN_NAME}_s1'
S2_NAME = f'{RUN_NAME}_s2'
S3_NAME = f'{RUN_NAME}_s3_final'
S1_BEST = os.path.join(WORK_DIR, 'runs', 'detect', S1_NAME, 'weights', 'best.pt')
S2_BEST = os.path.join(WORK_DIR, 'runs', 'detect', S2_NAME, 'weights', 'best.pt')
S3_BEST = os.path.join(WORK_DIR, 'runs', 'detect', S3_NAME, 'weights', 'best.pt')
YAML_PATH = os.path.join(WORK_DIR, 'dataset.yaml')

for d in [EXTRACT_PATH, WORK_DIR, DRIVE_RESULTS]:
    os.makedirs(d, exist_ok=True)

print(f'Model: {MODEL_SIZE} | Image: {IMG_SIZE} | Batch: {BATCH}')
print(f'Results: {DRIVE_RESULTS}')


In [ ]:
import zipfile, shutil

UAV_EXTRACT = os.path.join(EXTRACT_PATH, 'uav')
os.makedirs(UAV_EXTRACT, exist_ok=True)

if os.path.exists(UAV_ZIP_PATH):
    print(f'Extracting {UAV_ZIP_PATH} ...')
    with zipfile.ZipFile(UAV_ZIP_PATH, 'r') as zf:
        # Filter out __MACOSX junk
        members = [m for m in zf.namelist() if not m.startswith('__MACOSX')]
        zf.extractall(UAV_EXTRACT, members=members)
    print(f'Done. Extracted {len(members)} items')
else:
    raise FileNotFoundError(
        f'MTDC-UAV.zip not found at: {UAV_ZIP_PATH}\n'
        'Upload MTDC-UAV.zip to your Google Drive root folder first.'
    )


In [ ]:
import glob

def find_subdir(base, *patterns):
    """Search for a subdirectory matching any of the given patterns."""
    for pat in patterns:
        matches = glob.glob(os.path.join(base, pat))
        if matches:
            return matches[0]
    return None

# Auto-detect directory structure
UAV_IMG_DIR = find_subdir(UAV_EXTRACT, '**/images', '**/JPEGImages', '**/img*', '**/*images*')
UAV_XML_DIR = find_subdir(UAV_EXTRACT, '**/annotations', '**/Annotations', '**/xml*', '**/*labels*')

# If no subdirectories, images/XMLs might be at extract root
if UAV_IMG_DIR is None:
    jpgs = glob.glob(os.path.join(UAV_EXTRACT, '*.[jJ][pP]*[gG]'))
    if jpgs:
        UAV_IMG_DIR = UAV_EXTRACT
if UAV_XML_DIR is None:
    xmls = glob.glob(os.path.join(UAV_EXTRACT, '*.xml'))
    if xmls:
        UAV_XML_DIR = UAV_EXTRACT

print(f'Image directory: {UAV_IMG_DIR}')
print(f'XML directory:   {UAV_XML_DIR}')

assert UAV_IMG_DIR, 'Could not find image directory - check the zip contents'
assert UAV_XML_DIR, 'Could not find XML annotation directory'

# Count images
IMG_EXTS = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
uav_images = []
for ext in IMG_EXTS:
    uav_images.extend(glob.glob(os.path.join(UAV_IMG_DIR, ext)))
print(f'UAV images found: {len(uav_images)}')

assert len(uav_images) > 0, 'No images found'


In [ ]:
import xml.etree.ElementTree as ET
from PIL import Image

WORK_LABELS = os.path.join(WORK_DIR, 'uav_labels_raw')
os.makedirs(WORK_LABELS, exist_ok=True)

def load_voc_xml(xml_path):
    """Parse Pascal VOC XML, return list of [cx, cy, w, h] normalized to [0,1]."""
    try:
        root = ET.parse(xml_path).getroot()
        size_el = root.find('size')
        if size_el is not None:
            w_img = float(size_el.findtext('width', '0'))
            h_img = float(size_el.findtext('height', '0'))
        else:
            w_img = h_img = 0

        boxes = []
        for obj in root.findall('object'):
            bbox = obj.find('bndbox')
            if bbox is None:
                continue
            xmin = float(bbox.findtext('xmin', '0'))
            ymin = float(bbox.findtext('ymin', '0'))
            xmax = float(bbox.findtext('xmax', '0'))
            ymax = float(bbox.findtext('ymax', '0'))
            bw = xmax - xmin
            bh = ymax - ymin
            if bw <= 0 or bh <= 0:
                continue
            # If no size element, use max coordinate values as image dims
            if w_img == 0 or h_img == 0:
                w_img = max(w_img, xmax)
                h_img = max(h_img, ymax)
            cx = (xmin + xmax) / 2 / w_img
            cy = (ymin + ymax) / 2 / h_img
            nw = bw / w_img
            nh = bh / h_img
            boxes.append([cx, cy, nw, nh])
        return boxes
    except Exception as e:
        print(f'  [WARN] {os.path.basename(xml_path)}: {e}')
        return []

# Build image-size lookup
print('Scanning image dimensions...')
img_sizes = {}
for img_path in uav_images:
    try:
        with Image.open(img_path) as im:
            img_sizes[os.path.splitext(os.path.basename(img_path))[0]] = im.size
    except Exception:
        pass
print(f'  Got sizes for {len(img_sizes)} images')

# Convert XML to YOLO labels
xml_files = glob.glob(os.path.join(UAV_XML_DIR, '*.xml'))
converted = 0
for xml_path in xml_files:
    stem = os.path.splitext(os.path.basename(xml_path))[0]
    boxes = load_voc_xml(xml_path)

    label_path = os.path.join(WORK_LABELS, f'{stem}.txt')
    with open(label_path, 'w') as f:
        for box in boxes:
            cx, cy, nw, nh = box
            f.write(f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n')
    converted += 1

print(f'Converted {converted} XML annotations to YOLO format')


In [ ]:
import numpy as np
from PIL import Image

TILE_SIZE    = 640
TILE_OVERLAP = 0.40
MIN_VIS_FRAC = 0.30

os.makedirs(os.path.join(TILED_DIR, 'images'), exist_ok=True)
os.makedirs(os.path.join(TILED_DIR, 'labels'), exist_ok=True)

def tile_uav_image(img_path):
    """Split a large UAV image into overlapping tiles.
    Returns the number of tiles created."""
    try:
        img = np.array(Image.open(img_path).convert('RGB'))
    except Exception as e:
        print(f'  [SKIP] {os.path.basename(img_path)}: {e}')
        return 0

    H, W = img.shape[:2]
    stem = os.path.splitext(os.path.basename(img_path))[0]
    stride = int(TILE_SIZE * (1 - TILE_OVERLAP))

    # Load labels
    lbl_path = os.path.join(WORK_LABELS, f'{stem}.txt')
    labels = []
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    labels.append([float(x) for x in parts])

    tile_count = 0
    for y0 in range(0, H, stride):
        for x0 in range(0, W, stride):
            tile = img[y0:y0+TILE_SIZE, x0:x0+TILE_SIZE]
            th, tw = tile.shape[:2]

            # Skip tiny edge tiles
            if th < TILE_SIZE * 0.3 or tw < TILE_SIZE * 0.3:
                continue

            tile_name = f'{stem}_t{tile_count:04d}_U'

            # Save tile image
            Image.fromarray(tile).save(
                os.path.join(TILED_DIR, 'images', f'{tile_name}.jpg'))

            # Adjust labels to tile coordinates
            tile_labels = []
            for label in labels:
                cls_id, cx, cy, nw, nh = label
                # Convert normalized to pixel in full image
                ax = cx * W
                ay = cy * H
                aw = nw * W
                ah = nh * H
                # Shift to tile coordinates
                tx = ax - x0
                ty = ay - y0
                # Check visibility fraction
                ix1 = max(tx - aw/2, 0)
                iy1 = max(ty - ah/2, 0)
                ix2 = min(tx + aw/2, tw)
                iy2 = min(ty + ah/2, th)
                vis_area = max(0, ix2 - ix1) * max(0, iy2 - iy1)
                box_area = aw * ah
                if box_area > 0 and vis_area / box_area >= MIN_VIS_FRAC:
                    ncx = tx / TILE_SIZE
                    ncy = ty / TILE_SIZE
                    nbw = aw / TILE_SIZE
                    nbh = ah / TILE_SIZE
                    ncx = max(0.0, min(1.0, ncx))
                    ncy = max(0.0, min(1.0, ncy))
                    nbw = max(0.001, min(1.0, nbw))
                    nbh = max(0.001, min(1.0, nbh))
                    tile_labels.append(f'{int(cls_id)} {ncx:.6f} {ncy:.6f} {nbw:.6f} {nbh:.6f}')

            if tile_labels:
                lbl_out = os.path.join(TILED_DIR, 'labels', f'{tile_name}.txt')
                with open(lbl_out, 'w') as f:
                    f.write('\n'.join(tile_labels))

            tile_count += 1

    return tile_count

# Run tiling
print(f'Tiling {len(uav_images)} UAV images (640x640, {TILE_OVERLAP*100:.0f}% overlap)...')
total_tiles = 0
for i, img_path in enumerate(uav_images):
    tiles = tile_uav_image(img_path)
    total_tiles += tiles
    if (i + 1) % 50 == 0:
        print(f'  Processed {i+1}/{len(uav_images)} images, {total_tiles} tiles so far...')

print(f'SAHI tiling complete: {total_tiles} tiles from {len(uav_images)} images')


In [ ]:
import random, yaml

random.seed(42)

# Create dataset folder structure
for split in ('train', 'val', 'test'):
    for sub in ('images', 'labels'):
        os.makedirs(os.path.join(DATASET_DIR, split, sub), exist_ok=True)

# Collect all tile image-label pairs
tile_imgs = glob.glob(os.path.join(TILED_DIR, 'images', '*.[jJ][pP]*[gG]'))
pairs = []
for img_path in tile_imgs:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(TILED_DIR, 'labels', f'{stem}.txt')
    if os.path.exists(lbl_path):
        pairs.append((img_path, lbl_path, stem))

random.shuffle(pairs)
n = len(pairs)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

splits = {
    'train': pairs[:n_train],
    'val':   pairs[n_train:n_train+n_val],
    'test':  pairs[n_train+n_val:],
}

for split_name, split_pairs in splits.items():
    for img_path, lbl_path, stem in split_pairs:
        ext = os.path.splitext(img_path)[1]
        shutil.copy2(img_path, os.path.join(DATASET_DIR, split_name, 'images', f'{stem}{ext}'))
        shutil.copy2(lbl_path, os.path.join(DATASET_DIR, split_name, 'labels', f'{stem}.txt'))

# Report
for split_name in ('train', 'val', 'test'):
    n_img = len(os.listdir(os.path.join(DATASET_DIR, split_name, 'images')))
    n_lbl = len(os.listdir(os.path.join(DATASET_DIR, split_name, 'labels')))
    print(f'  {split_name}: {n_img} images, {n_lbl} labels')

# Create dataset.yaml
dataset_config = {
    'path': DATASET_DIR,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'names': {0: 'maize_tassel'},
    'nc': 1,
}
yaml_path = os.path.join(WORK_DIR, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

# Backup yaml to Drive
shutil.copy2(yaml_path, os.path.join(DRIVE_RESULTS, 'dataset.yaml'))
print(f'Dataset YAML saved to: {yaml_path}')
print(f'Total tiles: {n} (train={n_train}, val={n_val}, test={n-n_train-n_val})')


In [ ]:
import threading, time, json
from ultralytics import YOLO

STATUS_FILE = os.path.join(DRIVE_RESULTS, 'CHECKPOINT_STATUS.txt')

def write_status(stage, epoch, note=''):
    try:
        with open(STATUS_FILE, 'w') as f:
            f.write(f'stage={stage}\nepoch={epoch}\nnote={note}\n')
            f.write(f'time={time.strftime("%Y-%m-%d %H:%M:%S")}\n')
    except Exception:
        pass

class YOLOTrainer(YOLO):
    """YOLO wrapper that auto-backs up weights to Drive during training."""

    def __init__(self, model_path, stage_name):
        super().__init__(model_path)
        self.stage_name = stage_name

    def train(self, **kwargs):
        # Add epoch-end callback for backup
        def on_epoch_end(trainer):
            epoch = trainer.epoch + 1
            if epoch % 5 == 0:
                wdir = os.path.join(trainer.save_dir, 'weights')
                for fn in ['best.pt', 'last.pt']:
                    src = os.path.join(wdir, fn)
                    if os.path.exists(src):
                        dst = os.path.join(DRIVE_RESULTS, f'{self.stage_name}_{fn}')
                        shutil.copy2(src, dst)
                        if fn == 'last.pt':
                            shutil.copy2(src, os.path.join(DRIVE_RESULTS, 'LATEST_last.pt'))
                        if fn == 'best.pt':
                            shutil.copy2(src, os.path.join(DRIVE_RESULTS, 'LATEST_best.pt'))
                write_status(self.stage_name, epoch, f'auto-backup epoch {epoch}')

        self.add_callback('on_train_epoch_end', on_epoch_end)
        return super().train(**kwargs)

# Start background sync thread (every 10 min)
_stop_bg = threading.Event()

def bg_sync():
    while not _stop_bg.is_set():
        time.sleep(600)
        try:
            for stage in os.listdir(os.path.join(WORK_DIR, 'runs', 'detect')):
                for fn in ['best.pt', 'last.pt']:
                    src = os.path.join(WORK_DIR, 'runs', 'detect', stage, 'weights', fn)
                    if os.path.exists(src):
                        shutil.copy2(src, os.path.join(DRIVE_RESULTS, f'{stage}_{fn}'))
        except Exception:
            pass

bg_thread = threading.Thread(target=bg_sync, daemon=True)
bg_thread.start()

print('Backup system active (every 5 epochs + every 10 min background)')


In [ ]:
import torch

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print(f'Device: {"GPU" if DEVICE == "0" else "CPU"}')

print(f'\nStage 1: Backbone freeze warmup (30 epochs)')
print(f'  Model: {MODEL_SIZE}')
print(f'  Data:  MTDC-UAV tiles')
print(f'  Image: {IMG_SIZE}px')
print(f'  Optim: AdamW, lr=0.002, freeze=5')

model = YOLOTrainer(f'{MODEL_SIZE}.pt', S1_NAME)
results = model.train(
    data=YAML_PATH,
    epochs=30,
    imgsz=IMG_SIZE,
    batch=BATCH,
    freeze=5,
    optimizer='AdamW',
    lr0=0.002,
    device=DEVICE,
    name=S1_NAME,
    project=os.path.join(WORK_DIR, 'runs', 'detect'),
    exist_ok=True,
)

# Save to Drive
for fn in ['best.pt', 'last.pt']:
    src = os.path.join(WORK_DIR, 'runs', 'detect', S1_NAME, 'weights', fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_RESULTS, f'{S1_NAME}_{fn}'))

write_status(S1_NAME, 30, 'Stage 1 complete')
print('Stage 1 complete - weights saved to Drive')


In [ ]:
print(f'\nStage 2: Full training (80 epochs)')
print(f'  Starting from: Stage 1 best.pt')
print(f'  Optim: auto (MuSGD), lr=0.01')
print(f'  Augmentation: mixup=0.05, copy-paste')

model = YOLOTrainer(S1_BEST, S2_NAME)
results = model.train(
    data=YAML_PATH,
    epochs=80,
    imgsz=IMG_SIZE,
    batch=BATCH,
    optimizer='auto',
    lr0=0.01,
    mixup=0.05,
    copy_paste=0.1,
    device=DEVICE,
    name=S2_NAME,
    project=os.path.join(WORK_DIR, 'runs', 'detect'),
    exist_ok=True,
)

for fn in ['best.pt', 'last.pt']:
    src = os.path.join(WORK_DIR, 'runs', 'detect', S2_NAME, 'weights', fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_RESULTS, f'{S2_NAME}_{fn}'))

write_status(S2_NAME, 80, 'Stage 2 complete')
print('Stage 2 complete - weights saved to Drive')


In [ ]:
print(f'\nStage 3: High-res fine-tuning (80 epochs, 1280px)')
print(f'  Starting from: Stage 2 best.pt')
print(f'  Image size: 1280px')
print(f'  Batch: {max(4, BATCH // 4)}')

model = YOLOTrainer(S2_BEST, S3_NAME)
results = model.train(
    data=YAML_PATH,
    epochs=80,
    imgsz=1280,
    batch=max(4, BATCH // 4),
    optimizer='auto',
    lr0=0.005,
    device=DEVICE,
    name=S3_NAME,
    project=os.path.join(WORK_DIR, 'runs', 'detect'),
    exist_ok=True,
)

BEST_WEIGHTS = os.path.join(WORK_DIR, 'runs', 'detect', S3_NAME, 'weights', 'best.pt')

for fn in ['best.pt', 'last.pt']:
    src = os.path.join(WORK_DIR, 'runs', 'detect', S3_NAME, 'weights', fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_RESULTS, f'{S3_NAME}_{fn}'))

write_status(S3_NAME, 80, 'Stage 3 complete - ALL TRAINING DONE')
print('Stage 3 complete - final model trained!')
print(f'Best weights at: {BEST_WEIGHTS}')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

print('Evaluating model on test set...')
model_eval = YOLO(BEST_WEIGHTS)

# Detection metrics
metrics = model_eval.val(
    data=YAML_PATH,
    split='test',
    imgsz=IMG_SIZE,
    conf=EVAL_CONF_THR,
    iou=TRAIN_IOU,
    device=DEVICE,
)

print(f'mAP@0.5:      {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')

# Counting accuracy
print('Computing counting accuracy...')
test_dir = os.path.join(DATASET_DIR, 'test')
results_all = model_eval.predict(
    os.path.join(test_dir, 'images'),
    conf=0.25,
    device=DEVICE,
    verbose=False,
)

gt_counts = []
pred_counts = []
for r in results_all:
    stem = os.path.splitext(os.path.basename(r.path))[0]
    lbl_path = os.path.join(test_dir, 'labels', f'{stem}.txt')
    gt = 0
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            gt = sum(1 for line in f if line.strip())
    pred = len(r.boxes) if r.boxes else 0
    gt_counts.append(gt)
    pred_counts.append(pred)

gt = np.array(gt_counts, dtype=float)
pred = np.array(pred_counts, dtype=float)
diff = pred - gt
mae = float(np.mean(np.abs(diff)))
rmse = float(np.sqrt(np.mean(diff ** 2)))
gt_var = float(np.sum((gt - gt.mean()) ** 2))
r2 = float(1 - np.sum(diff ** 2) / (gt_var + 1e-9)) if gt_var > 1e-9 else float('nan')

print(f'MAE:  {mae:.2f} tassels')
print(f'RMSE: {rmse:.2f} tassels')
print(f'R2:   {r2:.4f}')

# Plot
plt.figure(figsize=(7, 6))
plt.scatter(gt, pred, alpha=0.5, color='#2E7D32')
max_val = max(gt.max(), pred.max(), 1)
plt.plot([0, max_val], [0, max_val], 'r--', label='Perfect')
plt.xlabel('Ground Truth Count')
plt.ylabel('Predicted Count')
plt.title(f'Counting Accuracy (MAE={mae:.1f}, R2={r2:.3f})')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_RESULTS, 'evaluation.png'), dpi=150)
plt.show()
print('Evaluation saved to Drive')


In [ ]:
import matplotlib.patches as patches
from PIL import Image
import random

model_pred = YOLO(BEST_WEIGHTS)
test_imgs = glob.glob(os.path.join(DATASET_DIR, 'test', 'images', '*.[jJ][pP]*[gG]'))
samples = random.sample(test_imgs, min(6, len(test_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, (ax, img_path) in enumerate(zip(axes.flat, samples)):
    result = model_pred.predict(img_path, conf=0.25, device=DEVICE, verbose=False)[0]
    img = Image.open(img_path)
    ax.imshow(img)
    count = 0
    if result.boxes is not None:
        for box in result.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = box
            ax.add_patch(patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                fill=False, edgecolor='#00FF00', linewidth=1.5))
        count = len(result.boxes)
    ax.set_title(f'{os.path.basename(img_path)} ({count} tassels)', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_RESULTS, 'inference_demo.png'), dpi=150)
plt.show()
print('Inference demo saved to Drive')


In [ ]:
from google.colab import files

# Copy final outputs to Drive
outputs = [
    (BEST_WEIGHTS, 'best.pt'),
    (os.path.join(WORK_DIR, 'runs', 'detect', S1_NAME, 'results.png'), 'stage1_curves.png'),
    (os.path.join(WORK_DIR, 'runs', 'detect', S2_NAME, 'results.png'), 'stage2_curves.png'),
    (os.path.join(WORK_DIR, 'runs', 'detect', S3_NAME, 'results.png'), 'stage3_curves.png'),
]

saved = 0
for src, dst in outputs:
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_RESULTS, dst))
        print(f'  [OK] {dst}')
        saved += 1
    else:
        print(f'  [MISSING] {src}')

print(f'\n{saved}/{len(outputs)} items saved to Drive')

# Download best.pt
print('\nDownloading best.pt to your PC...')
try:
    files.download(BEST_WEIGHTS)
except Exception as e:
    print(f'Download failed: {e}')
    print(f'Manually download from Drive: {DRIVE_RESULTS}/best.pt')

_bg_thread._stop_event.set() if hasattr(_bg_thread, '_stop_event') else None
print('\nTraining complete!')


In [ ]:
# === RESUME FROM CRASH (only run if Colab disconnected) ===
# Set which stage to resume

RESUME_STAGE = 'S3'  # Change to S1, S2, or S3

stage_map = {'S1': S1_NAME, 'S2': S2_NAME, 'S3': S3_NAME}
stage_name = stage_map[RESUME_STAGE]

# Look for checkpoint in Drive
candidates = [
    os.path.join(DRIVE_RESULTS, f'{stage_name}_last.pt'),
    os.path.join(DRIVE_RESULTS, 'LATEST_last.pt'),
    os.path.join(DRIVE_RESULTS, f'{stage_name}_best.pt'),
    os.path.join(DRIVE_RESULTS, 'LATEST_best.pt'),
]

checkpoint = None
for path in candidates:
    if os.path.exists(path):
        checkpoint = path
        print(f'Found: {path}')
        break

if checkpoint:
    local_dir = os.path.join(WORK_DIR, 'runs', 'detect', stage_name, 'weights')
    os.makedirs(local_dir, exist_ok=True)
    shutil.copy2(checkpoint, os.path.join(local_dir, 'last.pt'))
    print(f'Restored. Now re-run the training cell for {RESUME_STAGE}.')
else:
    print('No checkpoint found. Start from Cell 9 (Stage 1).')


## Training Complete - Next Steps

### 1. Download best.pt
The trained model is on your Google Drive: `maize_yolo26_results/best.pt`

### 2. Place it in the backend
```
Counting-Maize-Tassels-in-the-Wild-via-Deep-Neural-Network/
  backend/
    models/
      best.pt   <-- put it here
```

### 3. Restart the Flask backend
```bash
cd backend
python app.py
```

You should see: `AI Inference: YOLO model loaded - real detection enabled`

### 4. Test
Upload a maize field image through the web interface. The system will now use real YOLO26s detection instead of mock data. No code changes needed - `inference.py` handles everything automatically.
